In [ ]:
import warnings
from pathlib import Path
import matplotlib.pyplot as plt

import prism

from imagematerials.factory import ModelFactory, Sector
from imagematerials.model import (
    GenericMaterials,
    GenericStocks,
    MaterialIntensities,
    SharesInflowStocks
)
from imagematerials.vehicles.battery import ElectricVehicleBatteries, ElectricVehicleBatteries_TV, BatteryMaterials, LinkModule
from imagematerials.preprocessing import get_preprocessing_data
from imagematerials.electricity.preprocessing import get_preprocessing_data_evbattery

from imagematerials.electricity.constants import STANDARD_SCEN_EXTERNAL_DATA

path_current = Path().resolve()
path_base = path_current.parent #.parent # base path of the project -> image-materials
path_data = Path(path_base, "data", "raw")

In [ ]:
vhc_sector = get_preprocessing_data("vehicles", Path("..", "data", "raw"), 
                                    climate_policy_scenario_dir = None, 
                                    circular_economy_scenario_dirs = None)


In [ ]:
path_data

In [ ]:
scenario = path_data /"electricity"/ STANDARD_SCEN_EXTERNAL_DATA
ev_battery_sector = get_preprocessing_data("ev_battery", path_data, 
                                    climate_policy_scenario_dir = Path(path_data, "image", "SSP2_M_CP"), 
                                    circular_economy_scenario_dirs = None,
                                    cache = False, 
                                    standard_scenario = scenario)

In [ ]:
prep_data = get_preprocessing_data_evbattery_old(path_base=path_base, scenario = "SSP2_M_CP", year_start=1926, year_out=2100)
sec_evbattery = Sector("ev_batteries", prep_data, check_coordinates=False)

In [ ]:
scenario_list = {"SSP2_M_CP":("SSP2_M_CP", None),
                # "SSP2_narrow":("SSP2_narrow", "narrow"),
                 }

# scenario_base_path = Path(path_base) / 'circular_economy_scenarios'

time_start = 2000
time_end = 2060
complete_timeline = prism.Timeline(time_start, time_end, 1)
simulation_timeline = prism.Timeline(time_start, time_end, 1)

all_output = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    climate_policy_scenario_dir = Path(path_base, "image", climate_scen)
    # climate_policy_scenario_dir = Path(path_base, "IMAGE_CircoMod", climate_scen)
    circular_economy_scenario_dir = None #Path(path_base, "circular_economy_scenarios", circular_scen)
    scenario = path_base /"electricity"/ STANDARD_SCEN_EXTERNAL_DATA

    vhc_sector = get_preprocessing_data("vehicles", Path("..", "data", "raw"), 
                                    climate_policy_scenario_dir = None, 
                                    circular_economy_scenario_dirs = None)
    ev_battery_sector = get_preprocessing_data("ev_battery", Path("..", "data", "raw"),
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dir, cache = False, scenario = scenario)     
    elc_sector = get_preprocessing_data("electricity", Path("..", "data", "raw"),
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dir, cache = False, scenario = scenario) 
    # elc_sector is a list of preprocessing data for each electricity subsector
    

    factory = ModelFactory(
        [vhc_sector, sec_evbattery], complete_timeline
        ).add(GenericStocks, ["vehicles"]
        ).add(GenericMaterials, ["vehicles"]
        ).add(ElectricVehicleBatteries, ["ev_battery"], input_sources={
        "stock_by_cohort": "vehicles",
        "inflow": "vehicles",
        "outflow_by_cohort": "vehicles"}
        )
    model = factory.finish()

    # factory = ModelFactory(
    # elc_sector, complete_timeline
    # ).add(GenericStocks, ["elc_gen",
    #                      "elc_grid_lines",
    #                      "elc_grid_add",
    #                      "elc_stor_phs"
    #                       ]
    # ).add(SharesInflowStocks, ["elc_stor_other"],
    # ).add(MaterialIntensities, ["elc_gen",
    #                      "elc_grid_lines",
    #                      "elc_grid_add",
    #                      "elc_stor_phs",
    #                      "elc_stor_other"
    #                       ]
    # )
    # model = factory.finish()

    model.simulate(simulation_timeline)
    print(f"Finished {scen_id}")

list(model.ev_battery)

In [ ]:
sec_evbattery.coordinates.keys()

In [ ]:
time_start = 2000
time_end = 2060
complete_timeline = prism.Timeline(time_start, time_end, 1)
simulation_timeline = prism.Timeline(time_start, time_end, 1)

factory = ModelFactory(
    [vhc_sector, sec_evbattery], complete_timeline
    ).add(GenericStocks, ["vehicles"]
    ).add(GenericMaterials, ["vehicles"]
    ).add(ElectricVehicleBatteries, ["ev_batteries"], input_sources={
    "stock_by_cohort": "vehicles",
    "inflow": "vehicles",
    "outflow_by_cohort": "vehicles"}
    )
model = factory.finish()

warnings.filterwarnings("ignore")
model.simulate(simulation_timeline)

list(model.ev_batteries)

In [ ]:
model.ev_batteries['material_fractions']

In [ ]:
inflow_battery = model.ev_batteries["inflow_battery_kWh"].to_array()
inflow_battery

da = inflow_battery.sum(["Type", "Region"])
da

for bat in da.BatteryType.values:
    plt.plot(da.time, da.sel(BatteryType=bat), label=str(bat))
plt.xlabel("Year")
plt.ylabel("Inflow of EV batteries (kWh)")
plt.legend()
plt.title("Inflow of EV batteries")

In [ ]:
inflow_bat_mat = model.ev_batteries["inflow_battery_materials"].to_array().sum(["BatteryType", "Region", "Type"])

for mat in inflow_bat_mat.material.values:
    plt.plot(inflow_bat_mat.time, inflow_bat_mat.sel(material=mat), label=str(mat))
plt.xlabel("Year")
plt.ylabel("Inflow of material in EV batteries (kg)")
plt.legend()
plt.title("Inflow of materials in EV batteries")

In [ ]:
inflow_bat = model.ev_batteries['stock_battery_kWh_v2g'].sum(["BatteryType","Region"])
# t="test"
for t in inflow_bat.Type.values:
    plt.plot(inflow_bat.Time, inflow_bat.sel(Type=t), label=str(t)) #
plt.xlabel("Year")
plt.ylabel("stock of EV batteries (kWh)")
plt.legend(bbox_to_anchor=(1.02, 0.5), loc="center left")
plt.title("stock of EV batteries")

In [ ]:
inflow_vhc = model.vehicles["inflow"].to_array().sum("Region")#.loc[slice(2021,2029)]

for vhc in inflow_vhc.Type.values:
    plt.plot(inflow_vhc.time, inflow_vhc.sel(Type=vhc), label=str(vhc))
plt.xlabel("Year")
plt.ylabel("Inflow of vehicles (count)")
plt.legend(bbox_to_anchor=(1.02, 0.5), loc="center left")
plt.title("Inflow of vehicles")